# V01: Brain-driven Agent Harness — Offline Experiment

**Question.** Can the local pipeline turn PDF, web, and project-history fixtures into a reproducible research result with structured runtime logs, a local MLflow mirror, and a validated canonical manifest?

**Success criteria.** Three sources are registered, retrieval returns three hits, provenance completeness is 1.0, every required run artifact validates, and MLflow produces an explicit synced/deferred receipt. This notebook performs no network, paid API, GPU, or confirmation-set access.


In [ ]:
from __future__ import annotations

import json
import random
import shutil
import sys
from pathlib import Path

SEED = 7
random.seed(SEED)
RESEARCH = Path.cwd().resolve()
while RESEARCH != RESEARCH.parent and not (RESEARCH / '05_code' / 'src').is_dir():
    RESEARCH = RESEARCH.parent
assert (RESEARCH / '05_code' / 'src').is_dir(), RESEARCH
sys.path.insert(0, str(RESEARCH / '05_code' / 'src'))
WORKSPACE_ROOT = RESEARCH.parents[2]
MLFLOW_ROOT = WORKSPACE_ROOT / '01_Stores' / '00_myIS' / 'mlflow'
WORK = RESEARCH / '03_experiments' / 'V01_brain_drive_agent_demo' / 'notebook_workspace'
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
{'seed': SEED, 'research': str(RESEARCH), 'mlflow_root': str(MLFLOW_ROOT)}


## Predeclared plan

1. Create deterministic local fixtures for a PDF, a web record, and old project history.
2. Register and retrieve them through the QMD-compatible fixture Brain.
3. Execute synthesis inside the immutable Harness kernel.
4. Emit one structlog event to console/runtime and project milestones to progress JSONL.
5. Mirror metrics/artifacts into local MLflow, finalize the immutable manifest, and validate hashes.
6. Generate a paper-table row only from validated manifest + metrics artifacts.


In [ ]:
from myis_research.brain_drive import run_brain_drive_demo

output = run_brain_drive_demo(WORK, mlflow_root=MLFLOW_ROOT)
run_dir = Path(output['run_dir'])
{'run_dir': str(run_dir), 'report': output['report']}


## Validate the canonical bundle

The validator re-hashes every manifest-listed artifact, verifies JSONL structure and event sequence, and confirms that progress events are a subset of runtime events.


In [ ]:
from myis_research.harness import validate_run_bundle

validation = validate_run_bundle(run_dir)
required = {
    'prompt.json', 'flow.json', 'progress.jsonl', 'result.json', 'metrics.json',
    'runtime.jsonl', 'per_query_metrics.jsonl', 'validation_report.json', 'manifest.json'
}
assert required <= {path.name for path in run_dir.iterdir()}
metrics = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))
assert metrics == {'provenance_completeness': 1.0, 'retrieval_hit_count': 3.0, 'source_count': 3.0}
receipt = json.loads(next((run_dir / 'receipts').glob('mlflow-*.json')).read_text(encoding='utf-8'))
assert receipt['status'] in {'synced', 'sync_deferred'}
{'validation': validation, 'metrics': metrics, 'mlflow_receipt': receipt}


## Paper-table projection

This cell deliberately reads only the validated manifest and metrics file. It does not parse stdout, JSONL, or the MLflow UI.


In [ ]:
manifest = json.loads((run_dir / 'manifest.json').read_text(encoding='utf-8'))
assert validation['status'] == 'PASS'
paper_table = [{
    'run_id': manifest['identity']['run_id'],
    'arm': manifest['identity']['arm'],
    'split': manifest['inputs']['split'],
    'source_count': metrics['source_count'],
    'retrieval_hit_count': metrics['retrieval_hit_count'],
    'provenance_completeness': metrics['provenance_completeness'],
    'manifest_sha256': validation['manifest_sha256'],
}]
paper_table


## Result and next gate

If every assertion passes, the offline Brain → Harness → structlog → MLflow → manifest → paper-table path is operational. This is infrastructure evidence only; it does not open a scientific run. The next DAPFAM/SkillOpt/HarnessOpt development run still requires the declared R3 budget gate, and confirmation requires a separate R4 approval.
